<a href="https://colab.research.google.com/github/luquelab/Flyby-Denial/blob/main/FINALFINAL_Ngoc_Tao_draft_bioinformatic_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bioinformatics pipeline


## Install and import packages

In [ ]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import collections # Added for Counter

from google.colab import files
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.preprocessing import StandardScaler

# Install Biopython
!pip install biopython

from Bio import SeqIO # Import SeqIO for parsing FASTA files
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio import ExPASy
from Bio import SwissProt
from Bio import Align
from Bio.Seq import Seq # Added for creating Seq objects
from Bio.SeqRecord import SeqRecord # Added for creating SeqRecord objects

## Input data

### Option 1: Fetch FASTA from a Database (e.g., NCBI) using parameters
You can fetch protein sequences from a public database by providing specific search parameters. This method leverages Biopython's Entrez module to query databases like NCBI.


In [ ]:
from IPython.display import display, HTML
import ipywidgets as widgets
from Bio import Entrez

# --- Entrez Configuration ---
# IMPORTANT: NCBI's E-utilities require an email address for ALL requests.
# Please replace "your.email@example.com" with your actual email address below.
# This is a one-time setup for NCBI compliance and is not interactive.
Entrez.email = "kennedykreutzer15@gmail.com" # <--- REPLACE THIS WITH YOUR EMAIL

# Optionally, set an API key for higher request limits. (Highly Recommended)
# Get an API key from: https://ncbi.nlm.nih.gov/account/settings/
# You can store it in Colab's secrets and retrieve it like: userdata.get('NCBI_API_KEY')
# Entrez.api_key = "YOUR_NCBI_API_KEY" # Uncomment and replace if you have an API key

# Initialize the list to store all SeqRecord objects
if 'all_fasta_sequences' not in globals() or not isinstance(all_fasta_sequences, list):
    all_fasta_sequences = []

print("IMPORTANT: Ensure your Entrez.email is set in the code block above for NCBI access.")
print("An API key is highly recommended for higher request limits.")

# --- Widgets for user input ---
organism_widget = widgets.Text(description='Organism Name:', placeholder='e.g., Homo sapiens, Mus musculus', layout=widgets.Layout(width='500px'))
num_proteins_widget = widgets.IntText(description='Max Proteins:', value=100, min=1, max=10000, layout=widgets.Layout(width='200px'))
protein_type_widget = widgets.Text(description='Protein Type Filter:', value='RefSeq', placeholder='e.g., RefSeq, Ribosomal protein', layout=widgets.Layout(width='500px'))
fetch_button = widgets.Button(description='Fetch Sequences by Organism', button_style='success')
output_area = widgets.Output()

# Display widgets
display(organism_widget, num_proteins_widget, protein_type_widget, fetch_button, output_area)

def fetch_sequences_from_entrez(b):
    with output_area:
        output_area.clear_output()

        organism_name = organism_widget.value.strip()
        num_proteins = num_proteins_widget.value
        protein_type_filter = protein_type_widget.value.strip()

        if not organism_name:
            print("Please enter an Organism Name to fetch sequences.")
            return

        search_term = f"{organism_name}[Organism]"
        if protein_type_filter:
            search_term += f" AND {protein_type_filter}[filter]" # Assuming it's a filter, could be [Title] etc. depending on user input

        print(f"Searching for protein sequences for organism: {organism_name} (max {num_proteins} proteins, filter: {protein_type_filter})")

        try:
            # Search for IDs related to the organism
            search_handle = Entrez.esearch(db="protein", term=search_term, retmax=num_proteins)
            record = Entrez.read(search_handle)
            search_handle.close()
            id_list = record["IdList"]

            if not id_list:
                print(f"No protein sequences found for organism: {organism_name} with filter '{protein_type_filter}'. Please try a different name or broaden your search.")
                return

            print(f"Found {len(id_list)} protein IDs for {organism_name}. Fetching sequences...")

            # Fetch FASTA records for the found IDs
            fetch_handle = Entrez.efetch(db="protein", id=id_list, rettype="fasta", retmode="text")
            fasta_data = fetch_handle.read()
            fetch_handle.close()

            # Parse FASTA data and add to all_fasta_sequences
            newly_fetched_sequences = list(SeqIO.parse(io.StringIO(fasta_data), "fasta"))

            global all_fasta_sequences
            all_fasta_sequences = [] # Clear existing to avoid appending duplicates on re-run
            all_fasta_sequences.extend(newly_fetched_sequences)

            print(f"Successfully fetched {len(newly_fetched_sequences)} sequences.")
            print(f"Total sequences in all_fasta_sequences: {len(all_fasta_sequences)} SeqRecords.")

            # Print a few details for confirmation
            for i, seq_record in enumerate(all_fasta_sequences[:3]): # Show first 3 sequences
                print(f"  Sample: ID: {seq_record.id}, Length: {len(seq_record.seq)}")
                if len(seq_record.seq) > 50:
                    print(f"  Sequence preview: {str(seq_record.seq)[:50]}...\n")
                else:
                    print(f"  Sequence preview: {str(seq_record.seq)}\n")

        except Exception as e:
            print(f"An error occurred during fetching: {e}")

fetch_button.on_click(fetch_sequences_from_entrez)

print("\n--- Ready to fetch sequences by Organism Name ---")

### Option 2: Enter protein sequence manually

If you have a protein sequence readily available, you can directly paste it here. Make sure it's a raw protein sequence (amino acid letters only, no FASTA headers).

In [ ]:


# Ensure all_fasta_sequences exists and is a list
if 'all_fasta_sequences' not in globals() or not isinstance(all_fasta_sequences, list):
    all_fasta_sequences = []
    print("Initialized 'all_fasta_sequences' list for manually entered sequences.")

print("Please enter your protein sequences one by one. Press Enter on an empty line to finish.")

sequence_count = 0
while True:
    user_sequence_input = input(f"Enter protein sequence (or press Enter to finish): ")

    if not user_sequence_input.strip(): # Check for empty input to break the loop
        print("No more sequences to add. Finishing manual input.")
        break

    # Clean the input sequence (remove spaces, newlines, etc.)
    cleaned_sequence = user_sequence_input.strip().replace(' ', '').replace('\n', '').replace('\r', '')

    if cleaned_sequence:
        try:
            sequence_count += 1
            # Create a Seq object
            protein_seq = Seq(cleaned_sequence)
            # Create a SeqRecord object. Assign a generic ID if no specific one is provided.
            new_seq_record = SeqRecord(protein_seq, id=f"User_Input_Protein_{sequence_count}", description="Manually entered sequence")

            all_fasta_sequences.append(new_seq_record)
            print(f"  Added sequence {sequence_count}: ID: {new_seq_record.id}, Length: {len(new_seq_record.seq)}")
            if len(new_seq_record.seq) > 50:
                print(f"  Sequence preview: {str(new_seq_record.seq)[:50]}...\n")
            else:
                print(f"  Sequence preview: {str(new_seq_record.seq)}\n")

        except Exception as e:
            print(f"Error processing manual sequence input: {e}")
    else:
        print("No valid sequence entered. Please try again or press Enter to finish.")

print(f"Total sequences loaded from manual input: {sequence_count} SeqRecords. Current total in all_fasta_sequences: {len(all_fasta_sequences)}.")

# Note: Previous content for fetching from NCBI has been removed as per user request.

## Calculate Physicochemical Properties of Proteins

Using Biopython's `ProtParam` module, we can calculate various physicochemical properties of a protein sequence, such as its amino acid length, molecular weight, amino acid composition, isoelectric point, net charge, and extinction coefficient.

In [ ]:
sequences_for_analysis_prep = []
# This list will store dictionaries, each containing protein info, its ProteinAnalysis object, and its properties
protein_data_for_analysis = []

# --- Rely solely on all_fasta_sequences which should now contain both uploaded and fetched sequences ---
if 'all_fasta_sequences' in globals() and all_fasta_sequences: # Check in globals() as it's set by a previous cell
    print("Preparing protein sequences from 'all_fasta_sequences' for analysis.")
    for seq_record in all_fasta_sequences:
        sequences_for_analysis_prep.append({'id': seq_record.id, 'sequence': str(seq_record.seq)})
else:
    print("No protein sequences available for analysis. Please run the protein fetching or upload cells first.")


if sequences_for_analysis_prep:
    print(f"Found {len(sequences_for_analysis_prep)} protein(s) for physicochemical property analysis.")
    # Prepare the protein_data_for_analysis list with ProteinAnalysis objects
    for seq_info in sequences_for_analysis_prep:
        identifier = seq_info['id']
        protein_sequence_string = seq_info['sequence']

        if protein_sequence_string:
            analysed_seq_obj = ProteinAnalysis(protein_sequence_string)
            protein_data_for_analysis.append({
                'id': identifier,
                'sequence': protein_sequence_string,
                'analysis_obj': analysed_seq_obj,
                'properties': {} # Dictionary to store calculated properties for this protein
            })
        else:
            print(f"Skipping preparation for {identifier} due to empty sequence.")

print("\nProtein preparation complete. Proceeding to calculate individual properties in subsequent cells.")

### 1. Calculate Amino Acid Length

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    length = analysed_seq.length
    protein_data['properties']['length'] = length
    print(f"Protein: {identifier}, Length: {length}")

### 2. Calculate Molecular Weight

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    molecular_weight = analysed_seq.molecular_weight()
    protein_data['properties']['molecular_weight'] = molecular_weight
    print(f"Protein: {identifier}, Molecular Weight: {molecular_weight:.2f} Daltons")

### 3. Calculate Amino Acid Composition

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    protein_sequence_string = protein_data['sequence'] # Get sequence string directly

    # Use collections.Counter to get amino acid counts
    amino_acid_counts = collections.Counter(protein_sequence_string)

    total_aas = sum(amino_acid_counts.values())
    if total_aas == 0:
        amino_acid_composition_percent = {aa: 0.0 for aa in amino_acid_counts.keys()}
    else:
        amino_acid_composition_percent = {aa: count / total_aas for aa, count in amino_acid_counts.items()}

    protein_data['properties']['amino_acid_composition'] = amino_acid_composition_percent
    print(f"\nProtein: {identifier}, Amino Acid Composition (percentage):")
    for aa, percentage in amino_acid_composition_percent.items():
        print(f"  {aa}: {percentage:.2%}")

### 4. Calculate Isoelectric Point (pI)

In [ ]:
for protein_data in protein_data_for_analysis:
    identifier = protein_data['id']
    analysed_seq = protein_data['analysis_obj']
    isoelectric_point = analysed_seq.isoelectric_point()
    protein_data['properties']['isoelectric_point'] = isoelectric_point
    print(f"Protein: {identifier}, Isoelectric Point (pI): {isoelectric_point:.2f}")


### Summary of All Physicochemical Properties

In [ ]:


# Extract properties into a list of dictionaries suitable for DataFrame creation
summary_data = []
for protein_data in protein_data_for_analysis:
    protein_id = protein_data['id']
    props = protein_data['properties']
    row_data = {'id': protein_id}
    row_data.update(props) # Add all calculated properties
    summary_data.append(row_data)

if summary_data:
    properties_df = pd.DataFrame(summary_data)
    # Display properties for inspection
    display(properties_df)
else:
    print("No protein properties were collected for display.")


## Calculate Sequence Similarity

In [ ]:


# Initialize the aligner
aligner = Align.PairwiseAligner()
aligner.mode = 'global' # Use global alignment (Needleman-Wunsch)
aligner.match_score = 1
aligner.mismatch_score = -1
aligner.open_gap_score = -0.5
aligner.extend_gap_score = -0.1

# Or use a substitution matrix like BLOSUM62
aligner.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")

similarity_results = []
num_proteins = len(protein_data_for_analysis)

if num_proteins < 2:
    print("Need at least two protein sequences to calculate pairwise similarity.")
else:
    print(f"Calculating pairwise sequence similarity for {num_proteins} proteins...")
    for i in range(num_proteins):
        for j in range(i + 1, num_proteins):
            protein1_data = protein_data_for_analysis[i]
            protein2_data = protein_data_for_analysis[j]

            seq1_id = protein1_data['id']
            seq2_id = protein2_data['id']
            seq1 = protein1_data['sequence']
            seq2 = protein2_data['sequence']

            # Perform alignment
            alignments = aligner.align(seq1, seq2)
            # Take the best alignment score
            score = alignments.score

            similarity_results.append({
                'Protein_A_ID': seq1_id,
                'Protein_B_ID': seq2_id,
                'Similarity_Score': score
            })

    if similarity_results:
        similarity_df = pd.DataFrame(similarity_results)
        print("\nPairwise Sequence Similarity Results:")
        display(similarity_df)
    else:
        print("No similarity results to display.")


## Output for relationship between proteins
Next, we perform two types of relationship analysis:
1.  **Physicochemical Properties Clustering**: It uses the calculated length, molecular weight, and isoelectric point to cluster proteins, standardizing the data and then creating a dendrogram based on Euclidean distances.
2.  **Sequence Similarity Clustering**: It converts the previously calculated pairwise sequence similarity scores into distances and performs hierarchical clustering, displaying the relationships with another dendrogram.

In [ ]:


# Ensure protein_data_for_analysis is available and not empty
if 'protein_data_for_analysis' not in globals() or not protein_data_for_analysis:
    print("Error: No protein data found. Please run previous cells to load/analyze proteins.")
else:
    # Reconstruct properties_df from protein_data_for_analysis to ensure consistency
    summary_data = []
    for protein_data in protein_data_for_analysis:
        protein_id = protein_data['id']
        props = protein_data['properties']
        row_data = {'id': protein_id}
        row_data.update(props) # Add all calculated properties
        summary_data.append(row_data)

    if not summary_data:
        print("No protein properties were collected for analysis.")
        properties_df = pd.DataFrame() # Ensure it's defined even if empty
    else:
        properties_df = pd.DataFrame(summary_data)

    # Get ordered protein IDs for consistent indexing across analyses
    ordered_protein_ids = [p['id'] for p in protein_data_for_analysis]
    num_proteins = len(ordered_protein_ids)

    if num_proteins < 2:
        print("Need at least two protein sequences to perform clustering analyses.")
    else:
        # --- Helper function to get physicochemical distances ---
        def get_physchem_distances(properties_df, ordered_ids):
            # Ensure the properties_df is aligned with ordered_ids
            df_aligned = properties_df.set_index('id').reindex(ordered_ids).reset_index()
            # Select numerical features for clustering, excluding amino acid composition dictionary
            numeric_features_df = df_aligned[['length', 'molecular_weight', 'isoelectric_point']].copy()

            if numeric_features_df.empty or len(numeric_features_df) < 2:
                return None, None

            scaler = StandardScaler()
            scaled_features = scaler.fit_transform(numeric_features_df)
            return pdist(scaled_features, metric='euclidean'), ordered_ids

        # --- Helper function to get sequence distances ---
        def get_sequence_distances(similarity_df, ordered_ids):
            if similarity_df.empty or len(ordered_ids) < 2:
                return None, None

            num_proteins_ordered = len(ordered_ids)
            sequence_distance_matrix = np.zeros((num_proteins_ordered, num_proteins_ordered))
            id_to_index = {id_val: i for i, id_val in enumerate(ordered_ids)}

            max_score = similarity_df['Similarity_Score'].max()
            if max_score == 0:
                max_score = 1 # Avoid division by zero if all scores are 0

            for _, row in similarity_df.iterrows():
                # Only process if both proteins are in the ordered_ids list
                if row['Protein_A_ID'] in id_to_index and row['Protein_B_ID'] in id_to_index:
                    idx_a = id_to_index[row['Protein_A_ID']]
                    idx_b = id_to_index[row['Protein_B_ID']]
                    distance = max_score - row['Similarity_Score'] # Convert similarity to distance
                    sequence_distance_matrix[idx_a, idx_b] = distance
                    sequence_distance_matrix[idx_b, idx_a] = distance # Matrix is symmetric

            np.fill_diagonal(sequence_distance_matrix, 0)
            return squareform(sequence_distance_matrix), ordered_ids


        # --- 1. Relationship based on Physicochemical Properties ---
        print("\n--- Analyzing Relationships based on Physicochemical Properties ---")
        physchem_distances, physchem_labels = get_physchem_distances(properties_df, ordered_protein_ids)

        if physchem_distances is not None:
            physchem_linkage_matrix = linkage(physchem_distances, method='average')

            plt.figure(figsize=(12, 7))
            plt.title('Dendrogram of Proteins based on Physicochemical Properties')
            plt.xlabel('Protein ID')
            plt.ylabel('Distance')
            dendrogram(
                physchem_linkage_matrix,
                labels=physchem_labels,
                leaf_rotation=90.,
                leaf_font_size=8.,
            )
            plt.tight_layout()
            plt.show()
        else:
            print("Not enough protein entries with physicochemical properties to perform clustering (at least 2 needed).")

        # --- 2. Relationship based on Sequence Similarity ---
        print("\n--- Analyzing Relationships based on Sequence Similarity ---")
        # Check if similarity_df exists in globals and is not empty
        if 'similarity_df' in globals() and not similarity_df.empty:
            sequence_distances, sequence_labels = get_sequence_distances(similarity_df, ordered_protein_ids)
            if sequence_distances is not None:
                sequence_linkage_matrix = linkage(sequence_distances, method='average')

                plt.figure(figsize=(12, 7))
                plt.title('Dendrogram of Proteins based on Sequence Similarity')
                plt.xlabel('Protein ID')
                plt.ylabel('Distance')
                dendrogram(
                    sequence_linkage_matrix,
                    labels=sequence_labels,
                    leaf_rotation=90.,
                    leaf_font_size=8.,
                )
                plt.tight_layout()
                plt.show()
            else:
                print("Could not generate sequence distances. Check if similarity_df contains enough valid protein pairs or if enough proteins are loaded.")
        else:
            print("No sequence similarity data found or not enough proteins for clustering (at least 2 needed).")

        # --- 3. Relationship based on Combined Physicochemical Properties and Sequence Similarity ---
        print("\n--- Analyzing Relationships based on Combined Properties and Similarity ---")

        # Only proceed if both types of distances can be calculated for at least 2 proteins
        physchem_distances_combined, _ = get_physchem_distances(properties_df, ordered_protein_ids)
        sequence_distances_combined, _ = None, None # Initialize for the combined check

        if 'similarity_df' in globals() and not similarity_df.empty:
             sequence_distances_combined, _ = get_sequence_distances(similarity_df, ordered_protein_ids)

        if physchem_distances_combined is not None and sequence_distances_combined is not None:
            # Normalize distances to a 0-1 range before combining
            norm_physchem_dist = (physchem_distances_combined - physchem_distances_combined.min()) / (physchem_distances_combined.max() - physchem_distances_combined.min())
            norm_sequence_dist = (sequence_distances_combined - sequence_distances_combined.min()) / (sequence_distances_combined.max() - sequence_distances_combined.min())

            # Combine distances (e.g., by averaging)
            combined_distances = (norm_physchem_dist + norm_sequence_dist) / 2

            # Perform hierarchical clustering
            combined_linkage_matrix = linkage(combined_distances, method='average')

            # Plot dendrogram
            plt.figure(figsize=(12, 7))
            plt.title('Dendrogram of Proteins based on Combined Properties and Similarity')
            plt.xlabel('Protein ID')
            plt.ylabel('Distance')
            dendrogram(
                combined_linkage_matrix,
                labels=ordered_protein_ids,
                leaf_rotation=90.,
                leaf_font_size=8.,
            )
            plt.tight_layout()
            plt.show()
        else:
            print("Not enough data to perform combined analysis (at least 2 proteins with both physicochemical properties and sequence similarity data needed).")

### Download Results to CSV

To save the results of the physicochemical properties and sequence similarity analysis, you can download the respective DataFrames to CSV files. This ensures that your analysis outputs are preserved for future use or sharing.

In [ ]:
if 'properties_df' in globals() and not properties_df.empty:
    properties_df.to_csv('physicochemical_properties.csv', index=False)
    print("Physicochemical properties saved to 'physicochemical_properties.csv'")
else:
    print("No physicochemical properties DataFrame found to save.")

if 'similarity_df' in globals() and not similarity_df.empty:
    similarity_df.to_csv('sequence_similarity.csv', index=False)
    print("Sequence similarity results saved to 'sequence_similarity.csv'")
else:
    print("No sequence similarity DataFrame found to save.")

You can now locate and download these `.csv` files from the Colab file browser (left sidebar, folder icon).

## Prepare for next run

Finally, it is important to run this cell starting a new analysis without interference from previous runs. This cell provides a utility to clear all previously loaded and analyzed protein data from the current session.

In [ ]:
all_fasta_sequences = []
print("Data in 'all_fasta_sequences' has been cleared.")

protein_data_for_analysis = []
print("Data in 'protein_data_for_analysis' has been cleared.")

# Clear additional data structures for a fresh start
if 'properties_df' in globals():
    del properties_df
    print("'properties_df' has been cleared.")
if 'similarity_df' in globals():
    del similarity_df
    print("'similarity_df' has been cleared.")
